<a href="https://colab.research.google.com/github/mihikag-tech/UTD-Deep-Dive-AI-Project/blob/test/XGB_troubleshoot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import pickle

In [3]:
from sklearn.model_selection import train_test_split
from sklearn import tree
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [4]:
df = pd.read_csv("/content/Combined_dataset_model.csv")
df = pd.get_dummies(df, columns=["biome"], dtype=int)
df = df.drop(columns=['Unnamed: 0'])
print(df.columns)


Index(['county', 'land_area', 'tc_goal', 'treecanopy', 'tc_gap', 'priority_i',
       'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc', 'depratnorm',
       'health_nor', 'temp_norm', 'tes', 'tesctyscor', 'rank', 'rankgrpsz',
       'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp', 'Max_Temp',
       'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain', 'Max_Rain',
       'biome_Desert', 'biome_Forest', 'biome_Grassland'],
      dtype='object')


In [5]:
features = ['land_area', 'treecanopy', 'tc_gap',
       'priority_i', 'pctpocnorm', 'pctpovnorm', 'unemplnorm', 'dep_perc',
       'depratnorm', 'tes', 'tesctyscor', 'rank',
       'rankgrpsz', 'Mean_Temp', 'Median_Temp', 'STD_Temp', 'Min_Temp',
       'Max_Temp', 'Mean_Rain', 'Median_Rain', 'STD_Rain', 'Min_Rain',
       'Max_Rain', 'biome_Desert', 'biome_Forest', 'biome_Grassland']
target = ['health_nor']
X_df = df[features]
y_df = df[target]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size = 0.2, random_state = 42)

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
}

grid_search = GridSearchCV(
    XGBRegressor(random_state = 42, objective='reg:squarederror'),
    param_grid,
    cv = 5,
    scoring = 'r2'
)

grid_search.fit(X_train, y_train)

print("Best params :", grid_search.best_params_)
print("Best CV acc :", round(grid_search.best_score_, 4))
print("Test acc    :", round(grid_search.best_estimator_.score(X_test, y_test), 4))

Best params : {'learning_rate': 0.2, 'max_depth': 4, 'n_estimators': 300}
Best CV acc : 0.6718
Test acc    : 0.674


In [ ]:
best_model = grid_search.best_params_
best_model

NameError: name 'grid_search' is not defined

In [7]:
model = XGBRegressor(learning_rate = 0.2, max_depth = 4, n_estimators = 300)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)


#model = pickle.load(open('xgb_model_pickle', 'rb'))
model_pred = model.predict(X_test)
print("R2 score:", r2_score(y_test, model_pred))



R2 score: 0.6676608324050903


In [11]:
# Define what each solution actually changes, and by how much
solution_effects = {
    "street_trees": {
        "treecanopy": 300, # adds 300 points of canopy
        "Mean_Temp": -90 # Changed from 'Mean Temp' to 'Mean_Temp'
    },

    "community_garden": {
        "treecanopy": 50,
        "Mean_Temp": -30  # Changed from 'Mean Temp' to 'Mean_Temp'
    }
}

In [14]:
def impact_calc(model, county, solutions_dict, feature_cols):
  impact_results = [] # To store results
  county_df_filtered = df[df["county"] == county]

  for index, row_data in county_df_filtered.iterrows():
    # Get prediction
    original_features = row_data[feature_cols]
    prediction_original = model.predict(pd.DataFrame([original_features]))[0]

    for solution_name, effects in solutions_dict.items(): # Iterate through solutions and their effects
      modified_features = original_features.copy() # Create a copy to modify
      for key, value in effects.items(): # Apply effects
        if key in modified_features.index:
          modified_features[key] += value
        else:
          print("Warning: Feature", key, "from solution", solution_name, "not found in model features. Skipping update for this feature.")

      # Predict with modified features
      prediction_modified = model.predict(pd.DataFrame([modified_features]))[0]

      # Calculate change
      pct_hbi_chg = (prediction_modified - prediction_original) / prediction_original if prediction_original != 0 else 0

      impact_results.append({
          "county": county,
          "row_index": index,
          "solution": solution_name,
          "original_health_nor_pred": prediction_original,
          "modified_health_nor_pred": prediction_modified,
          "pct_hbi_change": pct_hbi_chg
      })
  return impact_results


impact_calc(model, "Austin County", solution_effects, features)

[{'county': 'Austin County',
  'row_index': 114,
  'solution': 'street_trees',
  'original_health_nor_pred': np.float32(0.45750242),
  'modified_health_nor_pred': np.float32(0.6108859),
  'pct_hbi_change': np.float32(0.3352627)},
 {'county': 'Austin County',
  'row_index': 114,
  'solution': 'community_garden',
  'original_health_nor_pred': np.float32(0.45750242),
  'modified_health_nor_pred': np.float32(0.6108859),
  'pct_hbi_change': np.float32(0.3352627)},
 {'county': 'Austin County',
  'row_index': 115,
  'solution': 'street_trees',
  'original_health_nor_pred': np.float32(0.39504886),
  'modified_health_nor_pred': np.float32(0.48056418),
  'pct_hbi_change': np.float32(0.21646771)},
 {'county': 'Austin County',
  'row_index': 115,
  'solution': 'community_garden',
  'original_health_nor_pred': np.float32(0.39504886),
  'modified_health_nor_pred': np.float32(0.48056418),
  'pct_hbi_change': np.float32(0.21646771)},
 {'county': 'Austin County',
  'row_index': 116,
  'solution': 'stre

In [ ]:
def impact_calc(model, county, solutions, feature_cols):
  impact_df = []
  county_df = df[df["county"] == county]
  for row in county_df:
    prediction = model.predict(row)
    for solution in solution_effects[solution].items():
      for key, value in solution_effects[solution].items():
        county_df[key] = county_df[key] + value

    update_prediction = model.predict(row)
    pct_hbi_chg = update_prediction/prediction
    impact_df.concat(county, solution, pct_hbi_chg)
  return impact_df


test = impact_calc(model, "Austin County", solution_effect, )

In [ ]:
print(solution_effects['street_trees'].items())

dict_items([('treecanopy', 300), ('Mean Temp', -90)])
